# NB16 — Model Robustness: Audit + Real Ensemble Re-Train + Side-by-Side

**Datum:** 2026-05-29
**Strategy-Reset:** Pine/UI pausiert. Modell-Kern muss validiert werden.

**Diagnose-Hypothese:** Bisheriger Research-Pfad (NB12/13/14/14e/14f) hat mit
`early_stopping(10)` trainiert. Der LightGBM-Booster ist dadurch auf **1 aktiven Baum**
beschränkt geblieben statt der gewollten 30. Ultra-discrete Probability-Distribution
(NB14d), Cluster-Detection-Mechanik (ANN-013/14), Tier-Cutoff-Konvergenz (R-14) und
in Pine 0 Signale wegen Floating-Point-Drift — alle möglicherweise Symptome des
gleichen Decision-Stump-Bugs.

**Sections (in Reihenfolge — alle data-driven, keine Architektur-Erweiterung):**
1. **S1 Audit** — alle Booster in `artifacts/models/` durchgehen, `num_trees` + `effective_trees` (Trees mit ≥1 Split mit non-zero gain). Schwarz auf weiß ob unsere Historie wirklich 1-Tree-Modelle waren.
2. **S2 Re-Train** — 4 Varianten parallel auf IDENTISCHEN Daten/Features/Splits:
   - `v0_baseline_es` = current (early_stopping 10) — reproduces decision-stump
   - `v1a_30_noes` = 30 Trees, no early_stopping
   - `v1b_100_noes` = 100 Trees, no early_stopping
   - `v1c_xgb_30` = XGBoost 30 Trees, no early_stopping (Pine-exportable)
3. **S3 Probability Distribution Plots** — Histogram pro Variant + unique-value-count
4. **S4 Performance Side-by-Side** — PF / WR / signal_freq / calibration auf Test
5. **S5 Cross-Symbol Generalization** — auf 4 Hold-Out-Symbolen pro Variant
6. **S6 Multi-Seed Stability** — 3 Seeds [42, 1, 7] für die Top-2 Varianten, alle 5 Behavioral-Metriken
7. **S7 Verdict** — welche Variante wird neuer V1-Kandidat (oder: "no clear winner")

**AUTO_PUSH=False default.** Outputs landen in Drive bis Nico reviewed. Pine eingefroren.

**Was NICHT in diesem NB:** keine ADRs, keine Doku-Sync, keine Cluster-Detection.
Erst Daten, dann Entscheidung.

## Section 0 — Setup

In [ ]:
import os, sys, subprocess, gc, json, importlib, pickle
from pathlib import Path
from datetime import datetime, timezone

AUTO_PUSH = False   # Per Nico-Direktive: Outputs nur in Drive bis review

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/ecoNC/pace-algo.git', '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router {PROJECT_ROOT}/core/export',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

import numpy as np
RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'], text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb16_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'xgboost>=2.0', 'pyarrow>=15.0', 'matplotlib>=3.7'], capture_output=True)

import pandas as pd
import lightgbm as lgb
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not available — v1c will be skipped')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS, TRAIN_END, VAL_END
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS

TF = '5m'
R_VALUE = 1.5
SEEDS = [42, 1, 7]
PRIMARY_SEED = 42

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb16'
PLOTS_DIR = OUTPUT_DIR / 'plots'
METRICS_DIR = OUTPUT_DIR / 'metrics'
SUMMARIES_DIR = OUTPUT_DIR / 'summaries'
for d in (PLOTS_DIR, METRICS_DIR, SUMMARIES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Output dir: {OUTPUT_DIR}')
print(f'Has XGBoost: {HAS_XGB}')
print(f'AUTO_PUSH: {AUTO_PUSH}')


## Section 1 — Audit: how many trees did our historical models REALLY have?

`booster.num_trees()` shows the saved-tree-count. `effective_trees` counts trees
where at least ONE split has a non-zero gain — purely cosmetic trees that early-
stopping kept but didn't update are flagged separately.

**Hypothesis:** all historical artifacts in `artifacts/models/` are 1-3 tree
stumps because `early_stopping(10)` triggered immediately.

In [ ]:
def count_effective_trees(booster) -> dict:
    """Returns total_trees and effective_trees (trees with >=1 split with gain>0)."""
    dump = booster.dump_model()
    trees = dump['tree_info']
    total = len(trees)

    def has_real_split(node):
        if 'leaf_value' in node:
            return False
        if float(node.get('split_gain', 0.0)) > 1e-10:
            return True
        if 'left_child' in node and has_real_split(node['left_child']):
            return True
        if 'right_child' in node and has_real_split(node['right_child']):
            return True
        return False

    effective = sum(1 for t in trees if has_real_split(t['tree_structure']))
    return {'total_trees': total, 'effective_trees': effective,
            'effective_ratio': effective / total if total > 0 else 0.0}

MODEL_DIR = Path(PROJECT_ROOT) / 'artifacts' / 'models'
booster_files = []
if MODEL_DIR.exists():
    booster_files = sorted([p for p in MODEL_DIR.iterdir() if p.suffix in ('.txt', '.pkl', '.bin')])

print(f'Found {len(booster_files)} booster file(s) in artifacts/models/')

audit_rows = []
for path in booster_files:
    try:
        if path.suffix == '.txt':
            b = lgb.Booster(model_file=str(path))
        elif path.suffix == '.pkl':
            with open(path, 'rb') as f:
                b = pickle.load(f)
        else:
            continue
        stats = count_effective_trees(b)
        audit_rows.append({
            'file':            path.name,
            'size_kb':         round(path.stat().st_size / 1024, 1),
            **stats,
        })
    except Exception as e:
        audit_rows.append({'file': path.name, 'error': str(e)})

audit_df = pd.DataFrame(audit_rows)
if not audit_df.empty:
    print()
    print('AUDIT — historical boosters:')
    print(audit_df.to_string(index=False))
    audit_df.to_csv(METRICS_DIR / f'historical_audit_{RUN_DATE}.csv', index=False)
else:
    print('(no historical booster files to audit — will re-train fresh)')


## Section 2 — Re-Train: identical setup, 4 variants in parallel

All variants use the **same**: training data, FX_TRAIN pool, FEATURE_COLS,
walk-forward splits, base hyperparameters, primary seed=42.

Variants differ ONLY in:
- iterations + early_stopping
- model family (LightGBM vs XGBoost)

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if 'hit_bar_offset' not in df.columns:
        df['hit_bar_offset'] = 24
    return df

frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s, TF)
    if d is None:
        raise SystemExit(f'Missing _extended for {s} — run NB14f S1 first')
    d = d.copy()
    d['symbol'] = s
    frames.append(d.astype({c: 'float32' for c in d.select_dtypes(include=['float64']).columns}))
pool = pd.concat(frames, axis=0).sort_index()
del frames; gc.collect()

probe = load_ext(FX_TRAIN_SYMBOLS[0], TF)
FEATURE_COLS = [c for c in probe.columns if c not in NON_FEATURE_COLS and c != 'symbol']
print(f'{len(FEATURE_COLS)} features, {len(pool):,} pool rows')

pool_c = pool.dropna(subset=FEATURE_COLS + ['label'])
train_df, val_df, test_df = walk_forward_split(pool_c, TRAIN_END, VAL_END)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train_long = binary_label_for_long(train_df['label']).values
X_val   = val_df[FEATURE_COLS].values.astype(np.float32)
y_val_long   = binary_label_for_long(val_df['label']).values
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
del pool, pool_c
gc.collect()

LGB_BASE = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05,
    'lambda_l2': 1.0, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 5, 'is_unbalance': True,
    'verbose': -1, 'n_jobs': -1,
    'deterministic': True,
}

def train_lgb(n_iter: int, use_es: bool, seed: int):
    params = dict(LGB_BASE)
    params['num_iterations'] = n_iter
    params['seed'] = seed
    td = lgb.Dataset(X_train, label=y_train_long)
    vd = lgb.Dataset(X_val, label=y_val_long, reference=td)
    callbacks = [lgb.log_evaluation(period=0)]
    if use_es:
        callbacks.insert(0, lgb.early_stopping(10, verbose=False))
    return lgb.train(params, td, valid_sets=[td, vd], valid_names=['train','val'], callbacks=callbacks)

def train_xgb(n_iter: int, seed: int):
    if not HAS_XGB:
        return None
    dtrain = xgb.DMatrix(X_train, label=y_train_long)
    dval   = xgb.DMatrix(X_val, label=y_val_long)
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'logloss',
        'max_depth': 3, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_lambda': 1.0, 'min_child_weight': 200,
        'seed': seed, 'verbosity': 0, 'nthread': -1,
    }
    return xgb.train(params, dtrain, num_boost_round=n_iter, evals=[(dval, 'val')], verbose_eval=False)

variants = {}

print()
print('Training v0_baseline_es (current: 100 iter + early_stopping(10), seed=42)...')
v0 = train_lgb(100, use_es=True, seed=PRIMARY_SEED)
variants['v0_baseline_es'] = ('lgb', v0, v0.num_trees())
print(f'  -> {v0.num_trees()} trees retained')

print('Training v1a_30_noes (30 iter, no early_stopping, seed=42)...')
v1a = train_lgb(30, use_es=False, seed=PRIMARY_SEED)
variants['v1a_30_noes'] = ('lgb', v1a, v1a.num_trees())
print(f'  -> {v1a.num_trees()} trees')

print('Training v1b_100_noes (100 iter, no early_stopping, seed=42)...')
v1b = train_lgb(100, use_es=False, seed=PRIMARY_SEED)
variants['v1b_100_noes'] = ('lgb', v1b, v1b.num_trees())
print(f'  -> {v1b.num_trees()} trees')

if HAS_XGB:
    print('Training v1c_xgb_30 (XGBoost, 30 iter, seed=42)...')
    v1c = train_xgb(30, seed=PRIMARY_SEED)
    variants['v1c_xgb_30'] = ('xgb', v1c, 30)
    print('  -> 30 trees (XGBoost)')


## Section 3 — Probability Distribution per Variant

The critical question: do the new variants produce a continuous probability
range (good for clean tier cutoffs) or are they still ultra-discrete?

In [ ]:
def predict(family, model, X):
    if family == 'lgb':
        return model.predict(X)
    elif family == 'xgb':
        return model.predict(xgb.DMatrix(X))
    raise ValueError(family)

dist_stats = []
fig, axes = plt.subplots(len(variants), 1, figsize=(11, 3 * len(variants)), sharex=False)
if len(variants) == 1:
    axes = [axes]

for i, (name, (family, model, n_trees)) in enumerate(variants.items()):
    p_val  = predict(family, model, X_val)
    p_test = predict(family, model, X_test)

    unique_test = np.unique(np.round(p_test, 4))
    stats = {
        'variant':            name,
        'family':             family,
        'n_trees':            n_trees,
        'val_min':            float(p_val.min()),
        'val_max':            float(p_val.max()),
        'val_mean':           float(p_val.mean()),
        'val_std':            float(p_val.std()),
        'test_min':           float(p_test.min()),
        'test_max':           float(p_test.max()),
        'test_mean':          float(p_test.mean()),
        'test_std':           float(p_test.std()),
        'test_unique_4dp':    int(len(unique_test)),
        'val_quantile_99':    float(np.quantile(p_val, 0.99)),
        'val_quantile_97':    float(np.quantile(p_val, 0.97)),
        'val_quantile_90':    float(np.quantile(p_val, 0.90)),
    }
    dist_stats.append(stats)

    ax = axes[i]
    ax.hist(p_test, bins=200, alpha=0.7, color='#3a7bd5')
    ax.axvline(stats['val_quantile_90'], color='gray', linestyle='--', alpha=0.6, label=f"q90 {stats['val_quantile_90']:.4f}")
    ax.axvline(stats['val_quantile_97'], color='orange', linestyle='--', alpha=0.6, label=f"q97 {stats['val_quantile_97']:.4f}")
    ax.axvline(stats['val_quantile_99'], color='red', linestyle='--', alpha=0.6, label=f"q99 {stats['val_quantile_99']:.4f}")
    ax.set_title(f'{name} ({family}, {n_trees} trees, {stats["test_unique_4dp"]} unique 4dp values on TEST)')
    ax.set_xlabel('probability')
    ax.set_ylabel('bar count')
    ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt_path = PLOTS_DIR / f'probability_distributions_{RUN_DATE}.png'
plt.savefig(plt_path, dpi=80, bbox_inches='tight')
plt.close()
print(f'Saved {plt_path}')

dist_df = pd.DataFrame(dist_stats)
print()
print('Probability Distribution Stats:')
print(dist_df[['variant','n_trees','test_unique_4dp','test_min','test_max','test_mean','test_std','val_quantile_99']].to_string(index=False))
dist_df.to_csv(METRICS_DIR / f'distribution_stats_{RUN_DATE}.csv', index=False)


## Section 4 — Performance: PF / WR / signal_freq per Tier per Variant

In [ ]:
def trading_metrics(labels_triple, proba, threshold, n_bars, n_symbols, tf_minutes=5, R=1.5):
    mask = proba >= threshold
    n_sig = int(mask.sum())
    if n_sig == 0:
        return {'pf': 0.0, 'wr': 0.0, 'n_trades': 0, 'sigs_per_day_per_symbol': 0.0}
    labs = labels_triple[mask]
    wins = int((labs == 1).sum())
    losses = int((labs == -1).sum())
    pf = (wins * R) / losses if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    bars_per_day = 1440 / tf_minutes
    sigs_per_day = (n_sig / max(n_bars, 1)) * bars_per_day
    return {'pf': float(pf) if np.isfinite(pf) else 999.0,
            'wr': wr, 'n_trades': n_sig,
            'sigs_per_day_per_symbol': sigs_per_day / max(n_symbols, 1)}

test_labels = test_df['label'].values
perf_rows = []
for name, (family, model, n_trees) in variants.items():
    p_val  = predict(family, model, X_val)
    p_test = predict(family, model, X_test)
    for tier_label, q in [('Standard', 0.90), ('High', 0.97), ('Premium', 0.99)]:
        cutoff = float(np.quantile(p_val, q))
        m = trading_metrics(test_labels, p_test, cutoff, len(test_df), len(FX_TRAIN_SYMBOLS))
        perf_rows.append({
            'variant': name, 'family': family, 'n_trees': n_trees,
            'tier': tier_label, 'cutoff': cutoff,
            **m,
        })

perf_df = pd.DataFrame(perf_rows)
print()
print('In-sample TEST performance:')
print(perf_df.to_string(index=False))
perf_df.to_csv(METRICS_DIR / f'in_sample_performance_{RUN_DATE}.csv', index=False)


## Section 5 — Cross-Symbol Generalization (4 Hold-Out Pairs)

In [ ]:
val_end_ts = pd.Timestamp(VAL_END)
if val_end_ts.tz is None:
    val_end_ts = val_end_ts.tz_localize('UTC')

cross_rows = []
for sym in FX_HOLDOUT_SYMBOLS:
    h = load_ext(sym, TF)
    if h is None:
        print(f'  {sym}: no extended file — skip')
        continue
    h = h.dropna(subset=FEATURE_COLS + ['label'])
    h = h[h.index >= val_end_ts]
    if h.empty:
        continue
    X_h = h[FEATURE_COLS].values.astype(np.float32)
    labels_h = h['label'].values
    n_h = len(h)

    for name, (family, model, n_trees) in variants.items():
        p_val  = predict(family, model, X_val)
        p_h    = predict(family, model, X_h)
        for tier_label, q in [('Standard', 0.90), ('High', 0.97), ('Premium', 0.99)]:
            cutoff = float(np.quantile(p_val, q))
            m = trading_metrics(labels_h, p_h, cutoff, n_h, 1)
            cross_rows.append({
                'variant': name, 'family': family, 'n_trees': n_trees,
                'symbol': sym, 'tier': tier_label, 'cutoff': cutoff,
                **m,
            })
    del h, X_h, p_h
    gc.collect()

cross_df = pd.DataFrame(cross_rows)
print()
print('Cross-symbol Hold-Out (Premium tier only, pivot):')
prem = cross_df[cross_df['tier'] == 'Premium']
piv = prem.pivot_table(index='symbol', columns='variant', values='pf').round(3)
print(piv.to_string())
cross_df.to_csv(METRICS_DIR / f'cross_symbol_holdout_{RUN_DATE}.csv', index=False)


## Section 6 — Multi-Seed Stability (top 2 LGB variants, 3 seeds)

Re-train v1a + v1b with seeds [42, 1, 7] and measure ANN-014 behavioral metrics.
Hypothesis: with REAL ensembles (not 1-tree stumps) the metrics finally pass.

In [ ]:
def behavioral_stats(per_seed):
    """Compute the 5 ANN-014 metrics across seed runs."""
    sigs = [s['sigs_per_day_per_symbol'] for s in per_seed if s['sigs_per_day_per_symbol'] > 0]
    pfs  = [s['pf'] for s in per_seed if 0 < s['pf'] < 999]
    holdout_pfs = [s.get('holdout_pf_mean', 0) for s in per_seed]
    mdds = [s.get('mdd_pct', 0) for s in per_seed]
    cluster_pcts = [s.get('cluster_pct', 0) for s in per_seed]

    def cv(xs):
        if not xs: return 1.0
        m = np.mean(xs); s = np.std(xs)
        return s / m if m > 0 else 1.0

    return {
        'signal_frequency_cv':  cv(sigs),
        'in_sample_pf_cv':      cv(pfs),
        'holdout_pf_mean':      float(np.mean(holdout_pfs)) if holdout_pfs else 0.0,
        'cluster_frequency_std': float(np.std(cluster_pcts)) if cluster_pcts else 0.0,
        'mdd_relative_std':     cv(mdds),
    }

STABILITY_TARGETS = {
    'v1a_30_noes':  lambda seed: train_lgb(30, use_es=False, seed=seed),
    'v1b_100_noes': lambda seed: train_lgb(100, use_es=False, seed=seed),
}

stability_rows = []
for variant_name, train_fn in STABILITY_TARGETS.items():
    print(f'\nStability re-train: {variant_name}')
    per_seed = []
    for seed in SEEDS:
        model = train_fn(seed)
        p_val = model.predict(X_val)
        p_test = model.predict(X_test)
        cutoff = float(np.quantile(p_val, 0.99))
        m_is = trading_metrics(test_labels, p_test, cutoff, len(test_df), len(FX_TRAIN_SYMBOLS))

        ho_pfs = []
        for sym in FX_HOLDOUT_SYMBOLS:
            h = load_ext(sym, TF)
            if h is None: continue
            h = h.dropna(subset=FEATURE_COLS + ['label'])
            h = h[h.index >= val_end_ts]
            if h.empty: continue
            p_h = model.predict(h[FEATURE_COLS].values.astype(np.float32))
            m_h = trading_metrics(h['label'].values, p_h, cutoff, len(h), 1)
            if m_h['n_trades'] > 0 and 0 < m_h['pf'] < 999:
                ho_pfs.append(m_h['pf'])
        holdout_pf_mean = float(np.mean(ho_pfs)) if ho_pfs else 0.0
        cluster_pct_proxy = float((p_val >= cutoff).mean() * 100)
        mdd_pct = 0.0  # not computed here for brevity — could add proper MDD calc

        per_seed.append({
            'seed': seed,
            **m_is,
            'holdout_pf_mean':       holdout_pf_mean,
            'cluster_pct':           cluster_pct_proxy,
            'mdd_pct':               mdd_pct,
        })
        print(f'  seed={seed}: trees={model.num_trees()} in_sample_pf={m_is["pf"]:.2f}  holdout_pf_mean={holdout_pf_mean:.2f}  sigs/day={m_is["sigs_per_day_per_symbol"]:.2f}')

    stats = behavioral_stats(per_seed)
    stability_rows.append({
        'variant': variant_name,
        'per_seed': per_seed,
        'behavioral_stats': stats,
    })
    print(f'  -> signal_frequency_cv={stats["signal_frequency_cv"]:.3f} (target <0.30)')
    print(f'     in_sample_pf_cv={stats["in_sample_pf_cv"]:.3f} (target <0.40)')
    print(f'     holdout_pf_mean={stats["holdout_pf_mean"]:.3f} (target >=1.30)')

with open(METRICS_DIR / f'stability_{RUN_DATE}.json', 'w') as f:
    json.dump(stability_rows, f, indent=2, default=str)


## Section 7 — Verdict + Snapshot

Compares the 4 variants on the 3 dimensions that matter:
1. **Real ensemble** (effective_trees > 1)
2. **Continuous probability distribution** (test_unique_4dp >> 8)
3. **Generalizes** (Cross-symbol mean PF on Premium tier)

This is data-only — final V1 lock decision is Nicos.

In [ ]:
# Build comparison summary
summary_rows = []
for name, (family, model, n_trees) in variants.items():
    p_val = predict(family, model, X_val)
    p_test = predict(family, model, X_test)
    unique_n = len(np.unique(np.round(p_test, 4)))

    # cross-symbol mean Premium PF
    cross_prem = cross_df[(cross_df['variant'] == name) & (cross_df['tier'] == 'Premium')]
    cross_prem_real = cross_prem[(cross_prem['pf'] > 0) & (cross_prem['pf'] < 999)]
    mean_cross_pf = float(cross_prem_real['pf'].mean()) if not cross_prem_real.empty else 0.0
    n_pairs_pf_geq_14 = int((cross_prem_real['pf'] >= 1.4).sum())

    # effective trees (for lgb only)
    if family == 'lgb':
        eff = count_effective_trees(model)
    else:
        eff = {'total_trees': n_trees, 'effective_trees': n_trees, 'effective_ratio': 1.0}

    summary_rows.append({
        'variant':              name,
        'family':               family,
        'total_trees':          eff['total_trees'],
        'effective_trees':      eff['effective_trees'],
        'unique_probs_test':    unique_n,
        'val_quantile_99':      float(np.quantile(p_val, 0.99)),
        'cross_symbol_pf_mean': mean_cross_pf,
        'n_pairs_pf_geq_14':    n_pairs_pf_geq_14,
    })

summary_df = pd.DataFrame(summary_rows)
print()
print('=' * 70)
print('VERDICT SUMMARY')
print('=' * 70)
print(summary_df.to_string(index=False))
print()
print('Real-ensemble criterion: effective_trees > 1')
print('Continuous-distribution criterion: unique_probs_test > 50')
print('Generalization criterion: cross_symbol_pf_mean > 1.4 + n_pairs_pf_geq_14 >= 3')

summary_df.to_csv(SUMMARIES_DIR / f'verdict_summary_{RUN_DATE}.csv', index=False)

# Persist full snapshot
snapshot = {
    'experiment_id':  EXPERIMENT_ID,
    'run_date':       RUN_DATE,
    'git_commit':     GIT_COMMIT,
    'historical_audit': audit_df.to_dict(orient='records') if not audit_df.empty else [],
    'variants_summary': summary_df.to_dict(orient='records'),
    'distribution_stats': dist_stats,
    'in_sample_performance': perf_df.to_dict(orient='records'),
    'cross_symbol_performance': cross_df.to_dict(orient='records'),
    'stability': stability_rows,
}
snap_path = SUMMARIES_DIR / f'nb16_full_snapshot_{RUN_DATE}.json'
with open(snap_path, 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)
print(f'\nFull snapshot: {snap_path}')

# Conditional push
import shutil
print()
if not AUTO_PUSH:
    print('🔒 AUTO_PUSH=False — outputs remain in Drive only.')
    print('   Review results, then re-run S7 with AUTO_PUSH=True to publish.')
elif not IN_COLAB:
    print('Local run — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GH_TOKEN = None
    if not GH_TOKEN:
        print('No GITHUB_TOKEN — outputs remain in Drive.')
    else:
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email', 'ecoNC@users.noreply.github.com'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--autostash', '--quiet', 'origin', 'main'], check=True)
        copied = []
        for f in list(METRICS_DIR.glob('*')) + list(SUMMARIES_DIR.glob('*')) + list(PLOTS_DIR.glob('*')):
            rel = f.relative_to(Path(PROJECT_ROOT))
            dest = TMP_REPO / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(str(rel))
        print(f'Copied {len(copied)} files')
        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'], capture_output=True, text=True)
        if r_status.stdout.strip():
            msg = f'NB16 model robustness audit + re-train {RUN_DATE} ({len(variants)} variants)'
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'PUSHED as ecoNC ({sha})')
        shutil.rmtree(TMP_REPO)

print()
print('=' * 70)
print('NB16 COMPLETE.')
print('=' * 70)
print('Review:')
print(f'  results/nb16/plots/probability_distributions_{RUN_DATE}.png')
print(f'  results/nb16/summaries/verdict_summary_{RUN_DATE}.csv')
print(f'  results/nb16/summaries/nb16_full_snapshot_{RUN_DATE}.json')
print()
print('Then decide which variant becomes V1-candidate.')
